In [0]:
# Silver layer: customers
# Source: b2b_ecommerce.bronze.customers (append-only bronze ingestion)
# Grain:  one row per customer_id and version (history via __START_AT/__END_AT)
# Approach: bronze.customers is append-only, so we stream it, clean +
#           standardize each row, and merge into b2b_ecommerce.silver.customers
#           with AUTO CDC (merge on customer_id, sequenced by updated_at,
#           SCD Type 2 = keep the full change history).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_CUSTOMERS = "b2b_ecommerce.bronze.customers"
# current companies live in the silver layer (same pipeline dataset)
SILVER_COMPANIES = "companies"


def clean_text(name, lower=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned + standardized stream of bronze.customers (cleaning + company FK check)
# ---------------------------------------------------------------------------
@dp.temporary_view(name="customers_clean_stream")
# Data quality validation (only checks that can actually catch bad rows)
@dp.expect_or_drop("customer_id_not_null", "customer_id IS NOT NULL")  # merge key must exist
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")   # CDC sequence must exist
@dp.expect("company_id_not_null", "company_id IS NOT NULL")
@dp.expect_or_fail("company_id_exists", "_company_id_match IS NOT NULL")  # FK: silver.companies current rows, enforced
@dp.expect("email_valid", "email IS NOT NULL AND email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'")
@dp.expect("birth_date_not_future", "birth_date IS NULL OR birth_date < current_date()")
@dp.expect("phone_format_valid", "phone IS NULL OR phone RLIKE '^[+][0-9]'")  # international format
def customers_clean_stream():
    # bronze.customers is append-only: every change arrives as a new row.
    # The AUTO CDC merge below turns each change into a new version in the
    # SCD Type 2 history (sequenced by updated_at).
    bronze = spark.readStream.table(BRONZE_CUSTOMERS)

    # Referential integrity: customers are linked to companies, so flag every
    # row whose company_id has no match in the CURRENT silver.companies rows
    # (__END_AT IS NULL; a company that exists in bronze but was dropped from
    # silver does not match). Stream-static join against the same pipeline's
    # companies dataset, distinct keys so the join cannot fan out. The flag
    # drives the enforced company_id_exists expectation (fails the update on
    # orphans) and is excluded from the silver table.
    companies = (
        spark.read.table(SILVER_COMPANIES)
        .filter("__END_AT IS NULL")
        .select(F.col("company_id").alias("_company_id_match"))
        .distinct()
    )
    checked = bronze.join(
        companies, bronze.company_id == companies._company_id_match, "left"
    )

    gender_raw = F.lower(F.trim(F.col("gender")))  # 'M'/'Male'/'m' -> m, 'F'/'female' -> f
    doc = F.trim(F.col("document_number"))

    return checked.select(
        # Column types: explicit casts keep the silver schema stable
        F.col("customer_id").cast("int").alias("customer_id"),
        F.col("company_id").cast("int").alias("company_id"),
        F.col("birth_date").cast("date").alias("birth_date"),
        # Standardization: gender -> 'M'/'F', blank or unknown -> NULL
        F.when(gender_raw.isin("m", "male"), F.lit("M"))
        .when(gender_raw.isin("f", "female"), F.lit("F"))
        .otherwise(F.lit(None).cast("string"))
        .alias("gender"),
        # Cleaning: trim whitespace; blank text -> NULL; emails lowercased
        clean_text("first_name"),
        clean_text("last_name"),
        clean_text("email", lower=True),
        clean_text("phone"),  # valid intl. numbers (+54/+55/+598...)
        clean_text("job_title"),
        F.col("is_active").cast("boolean").alias("is_active"),
        # Invalid-value handling: keep only 7-8 digit document numbers, else NULL
        F.when(doc.rlike(r"^[0-9]{7,8}$"), doc)
        .otherwise(F.lit(None).cast("string"))
        .alias("document_number"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        # updated_at drives the CDC merge ordering; rows without it are
        # dropped by the updated_at_not_null expectation above (same drop
        # policy as silver.products/orders - no created_at fallback).
        F.col("updated_at").cast("timestamp").alias("updated_at"),
        # FK match flag (expectation only; excluded from the silver target)
        F.col("_company_id_match"),
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.customers
#    Rows are merged by customer_id; when the same customer is appended again
#    with a newer updated_at, a new version is kept (full history).
#    Current state per customer = the row where __END_AT IS NULL.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="customers",
    comment=(
        "Silver customers: cleaned, standardized; full history per customer_id "
        "with __START_AT/__END_AT validity (SCD Type 2)"
    ),
)

dp.create_auto_cdc_flow(
    target="customers",
    source="customers_clean_stream",
    keys=["customer_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2,
    # updated_at itself is excluded from history tracking: an identical
    # re-send (same values, newer updated_at) updates in place instead of
    # opening a spurious version (same as the other silver tables).
    track_history_except_column_list=["updated_at"],
    except_column_list=["_company_id_match"],
)
